# Project Axiom: Young Link AI Training

Train a competitive Young Link bot using imitation learning on Slippi replays.

**Requirements:**
- GPU Runtime (Runtime > Change runtime type > T4 GPU)
- Upload training data when prompted

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/sangokp/project-axiom.git
%cd project-axiom
!git checkout young-link-axiom

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# Create data directory structure
!mkdir -p data/training_yl/Parsed
!mkdir -p data/training_yl/Raw
!mkdir -p models/imitation

In [ ]:
# Upload training data
# You need to upload TWO files:
# 1. meta.json - the metadata file
# 2. training_data.tar.gz - the Parsed directory contents

from google.colab import files
print("Upload meta.json first:")
uploaded = files.upload()
!mv meta.json data/training_yl/

In [ ]:
# Upload the parsed training data
print("Now upload training_data.tar.gz:")
uploaded = files.upload()
!tar -xzf training_data.tar.gz -C data/training_yl/Parsed/
!rm training_data.tar.gz
print(f"Parsed files: {len(list(open('data/training_yl/meta.json').read()))} bytes")
!ls data/training_yl/Parsed/ | wc -l

In [ ]:
# Verify data loaded correctly
import json
with open('data/training_yl/meta.json') as f:
    meta = json.load(f)
print(f"Training replays: {len(meta)}")
print(f"Characters: {set(p['character'] for m in meta for p in m['players'])}")

In [ ]:
# Optional: Connect to Weights & Biases for tracking
# Uncomment and add your API key if you want logging
# import os
# os.environ['WANDB_API_KEY'] = 'your-api-key-here'

# Or run in offline mode
import os
os.environ['WANDB_MODE'] = 'offline'

In [ ]:
# Start imitation training
# Tuned for sparse YL data on T4 GPU

!python scripts/train.py \
  --wandb.mode=offline \
  --config.tag=axiom_yl_colab_v1 \
  --config.policy.delay=18 \
  \
  --config.data.batch_size=128 \
  --config.data.unroll_length=64 \
  \
  --config.learner.learning_rate=5e-5 \
  --config.learner.reward_halflife=4 \
  \
  --config.network.name=tx_like \
  --config.network.tx_like.num_layers=2 \
  --config.network.tx_like.hidden_size=256 \
  --config.network.tx_like.ffw_multiplier=2 \
  \
  --config.policy.train_value_head=False \
  --config.value_function.train_separate_network=True \
  --config.value_function.separate_network_config=True \
  --config.value_function.network.name=tx_like \
  --config.value_function.network.tx_like.num_layers=1 \
  --config.value_function.network.tx_like.hidden_size=256 \
  --config.value_function.network.tx_like.ffw_multiplier=2 \
  \
  --config.controller_head.name=autoregressive \
  --config.controller_head.autoregressive.component_depth=2 \
  --config.controller_head.autoregressive.residual_size=64 \
  \
  --config.dataset.allowed_characters=younglink \
  --config.dataset.allowed_opponents=all \
  --config.dataset.data_dir=data/training_yl/Parsed \
  --config.dataset.meta_path=data/training_yl/meta.json \
  \
  --config.runtime.eval_every_n=1000 \
  --config.runtime.num_eval_steps=100 \
  --config.runtime.max_runtime=36000 \
  --config.runtime.log_interval=100 \
  --config.runtime.save_interval=300

In [ ]:
# Download trained model
!ls -la experiments/
# Find the latest checkpoint
import glob
checkpoints = glob.glob('experiments/*/latest.pkl')
if checkpoints:
    latest = max(checkpoints, key=lambda x: open(x, 'rb').read()[:100])
    print(f"Downloading: {latest}")
    files.download(latest)

In [ ]:
# Alternative: Zip all experiments and download
!zip -r trained_model.zip experiments/
files.download('trained_model.zip')